# Day " of Month 4  is Today!, we'll be looking at Entity Resolution in a Graph hence fixing the new problem Day 1 Created.

**The problem this day exists to solve**
Day 1 removed Month2 Day 24's `resolve_entity` entirely, the four-tier matcher whose tier 4 fallback caused all that silent over-merging. That fixed the bug, but it left something obvious behind: `write_entity_to_graph` does `MERGE (e:Entity {name:$entity_name, type: $entity_type})`, which mathces on **exact name text only**. So "Services revenue FY2024" and "Apple's Services segment revenue" would create two completely separate nodes, the exact opposite failure from Month 3's Day 1 traded over-merging for under-merging, and that trade was deliberate (a fragmented graph is easier to repair than a corrupted one), but it isn't the end state.

Today builds real resolution back in, but designed to use the thing a graph that has that MongoDB never did: **the relationship structure itself as evodence**, not just the entity's name in isolation.

# Coding! Three Signals, Combined, With a Real Abstain Option
**Signal 1, Vector similarity on entity names:** Store an embedding alongside every entity node (using the same `bge-m3` model Days 6-7 of Month 3 confirmed was already the right choice), and use cosine similarity to find candidates. This is genuinely better than Month 2's Tier1/Tier2 string-and-keyword matching, which could never see that "shareholder returns" and "capital return program" mean the same thing.

**Signal 2, LLM judgement, but only as a tiebreaker:** Called only when the first two signals disagree or land in an ambiguous middle band, rather than as the primary mechanism. This directly applies Month 3 Day 11's cost lesson, the expensive check runs last, on the small subset that actually needs it.

**A Real "Don't Know" Path:** The genuinely important addition. When the combined evidence is ambiguous, the correct action is **create a separate node and record a `POSSIBLY_SAME_AS` relationship between them**, not force a merge, not silently create an unconnected duplicate. A graph can hold "these might be the same thing, unresolved" as a real, queryable state. MongoDB's flat documents never could, which is precisely why it's resolver was forced into a binary merge-or-don't decision that eventually produced Tier 4.`
 

In [1]:
from dataclasses import dataclass
from dotenv import load_dotenv, find_dotenv
from groq import Groq
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import Settings
import os
from typing import Optional

Settings.embed_model = HuggingFaceEmbedding(
    model_name='BAAI/bge-m3'
)

load_dotenv(find_dotenv())

client = Groq()

# TOKEN USER TRACKER
@dataclass
class TokenUsage:
    prompt_tokens: int = 0
    completion_tokens: int = 0
    total_calls: int = 0

    def add(self, usage):
        self.prompt_tokens += usage.prompt_tokens
        self.completion_tokens += usage.completion_tokens
        self.total_calls += 1
    
    def cost_estimate(
            self,
            input_price_per_1m: float = 2.50,
            output_price_per_1m: float = 10.00
    ) -> float:
        input_cost = (self.prompt_tokens /1000000) * input_price_per_1m
        output_cost = (self.completion_tokens /1000000) * output_price_per_1m
        return round(input_cost + output_cost, 6)
    
    def report(self):
        print(f"\n📣 Token usage report")
        print(f"LLM calls : {self.total_calls}")
        print(f"Prompt tokens: {self.prompt_tokens}")
        print(f"Completion tokens: {self.completion_tokens}")
        print(f"Total tokens: {self.prompt_tokens + self.completion_tokens:,}")
        print(f"Est. cost (GPT-40 pricing): ${self.cost_estimate()}")


# global tracher that is set before each run
usage_tracker = TokenUsage()


# TOKEN aware LLM Caller

def tracked_llm_call(messages: list, system: str = "") -> str:
    """
    Wraps every Groq call so token usage is always captured.
    Drop-in replacement for direct client.chat.completions.create calls.
    
    """

    full_messages = []
    if system:
        full_messages.append({"role": "system", "content": system})
    full_messages.extend(messages)

    response = client.chat.completions.create(
        model = "openai/gpt-oss-120b",
        messages= full_messages,
        temperature= 0,
    )

    usage_tracker.add(response.usage)
    return response.choices[0].message.content

c:\Users\rodne\miniconda3\envs\rag-Ai\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Connecting to a real Graph database

from neo4j import GraphDatabase

# AuraDB free tier: get these three values from your Neo4j Aura console
# after creating a free instance — no local install needed
NEO4J_URI = os.getenv("NEO4J_URI")
NEO4J_USERNAME = os.getenv("NEO4J_USERNAME")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")

neo4j_driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))

with neo4j_driver.session() as session:
    result = session.run("RETURN 'connected' AS status")
    print(f"✅ Neo4j: {result.single()['status']}")

✅ Neo4j: connected


In [9]:
# Storing embeddings alongside Entity Nodes

#from sentence_transformers import SentenceTransformer
import numpy as np

#entity_embedder = SentenceTransformer("BAAI/bge-m3")

def embed_entity_name(name: str) -> list[float]:
    return Settings.embed_model._get_text_embedding(name)

def get_all_entities_with_embeddings(user_id: str) -> list[dict]:
    with neo4j_driver.session() as session:
        result = session.run(
            """
            MATCH (u:User {id: $user_id})-[:ASKED_ABOUT]->(e:Entity)
            RETURN e.name AS name, e.type AS type, e.embedding AS embedding
            """,
            user_id=user_id
        )
        return [dict(r) for r in result if r["embedding"] is not None]


In [10]:
# The Graph-Structure Signal MongoDB never had

def shared_neighbourhood_score(entity_a: str, entity_b: str, user_id: str) -> float:
    """
    Counts how much graph structure two entities share. This signal simply
    does not exist in a flat-document store — it comes from the shape of the
    graph, not from either entity's own text.
    """
    with neo4j_driver.session() as session:
        result = session.run(
            """
            MATCH (a:Entity {name: $a})-[:RAISED_BY_QUESTION]->(q:Question)
            MATCH (b:Entity {name: $b})-[:RAISED_BY_QUESTION]->(q)
            RETURN count(DISTINCT q) AS shared_questions
            """,
            a=entity_a, b=entity_b
        )
        shared = result.single()["shared_questions"]

        totals = session.run(
            """
            MATCH (a:Entity {name: $a})-[:RAISED_BY_QUESTION]->(qa:Question)
            WITH count(DISTINCT qa) AS total_a
            MATCH (b:Entity {name: $b})-[:RAISED_BY_QUESTION]->(qb:Question)
            RETURN total_a, count(DISTINCT qb) AS total_b
            """,
            a=entity_a, b=entity_b
        ).single()

    if not totals or (totals["total_a"] + totals["total_b"]) == 0:
        return 0.0
    # Jaccard-style overlap
    return shared / (totals["total_a"] + totals["total_b"] - shared)

In [11]:
# The LLM, as a tiebreaker only
import re

RESOLUTION_TIEBREAK_PROMPT = """You decide whether two analysis topics from a financial
document conversation refer to the SAME underlying thing, or to genuinely DIFFERENT things.

Respond in EXACTLY this format:
Verdict: <SAME or DIFFERENT or UNCLEAR>
Reason: <one sentence>

UNCLEAR is a valid, useful answer — use it when the two could plausibly be either.
"""

def llm_tiebreak(name_a: str, name_b: str) -> tuple[str, str]:
    raw = tracked_llm_call(
        messages=[{"role": "user", "content": f"Topic A: {name_a}\nTopic B: {name_b}"}],
        system=RESOLUTION_TIEBREAK_PROMPT
    )
    verdict_match = re.search(r"Verdict:\s*(SAME|DIFFERENT|UNCLEAR)", raw)
    reason_match = re.search(r"Reason:\s*(.+)", raw)
    return ((verdict_match.group(1) if verdict_match else "UNCLEAR"),
            (reason_match.group(1).strip() if reason_match else raw.strip()))

In [12]:
# The Resolver, Three signals, With a Real Abstain Path

HIGH_CONFIDENCE_THRESHOLD = 0.88   # calibrated in Cell 6 below, not guessed
LOW_CONFIDENCE_THRESHOLD = 0.70

def resolve_entity_in_graph(new_name: str, new_type: str, user_id: str) -> dict:
    """
    Returns one of three real outcomes:
      {"action": "merge",   "target": <existing name>}
      {"action": "create",  "target": None}
      {"action": "link",    "target": <existing name>}   <- the option MongoDB never had
    """
    existing = get_all_entities_with_embeddings(user_id)
    if not existing:
        return {"action": "create", "target": None, "reason": "No existing entities"}

    new_emb = np.array(embed_entity_name(new_name))
    scored = []
    for e in existing:
        sim = float(np.dot(new_emb, np.array(e["embedding"])))
        scored.append({"name": e["name"], "similarity": sim})
    scored.sort(key=lambda x: -x["similarity"])
    best = scored[0]

    print(f"  🔍 Best name-similarity candidate: '{best['name']}' ({best['similarity']:.3f})")

    if best["similarity"] >= HIGH_CONFIDENCE_THRESHOLD:
        return {"action": "merge", "target": best["name"],
                "reason": f"High name similarity ({best['similarity']:.3f})"}

    if best["similarity"] < LOW_CONFIDENCE_THRESHOLD:
        return {"action": "create", "target": None,
                "reason": f"Low similarity to everything ({best['similarity']:.3f})"}

    # Ambiguous band — bring in the other two signals
    neighbourhood = shared_neighbourhood_score(new_name, best["name"], user_id)
    print(f"  🔗 Shared graph neighbourhood: {neighbourhood:.3f}")

    verdict, reason = llm_tiebreak(new_name, best["name"])
    print(f"  🧑‍⚖️ LLM tiebreak: {verdict} — {reason}")

    if verdict == "SAME" and neighbourhood > 0:
        return {"action": "merge", "target": best["name"], "reason": f"LLM SAME + shared structure"}
    if verdict == "DIFFERENT":
        return {"action": "create", "target": None, "reason": f"LLM DIFFERENT: {reason}"}

    # Genuinely uncertain — link, don't guess. This is the whole point of today.
    return {"action": "link", "target": best["name"],
            "reason": f"Ambiguous (sim {best['similarity']:.3f}, LLM {verdict}) — linked, not merged"}

In [13]:
# Writing to the graph, honouring all three outcomes

def write_entity_resolved(user_id: str, entity_name: str, entity_type: str, question: str):
    decision = resolve_entity_in_graph(entity_name, entity_type, user_id)
    embedding = embed_entity_name(entity_name)

    with neo4j_driver.session() as session:
        if decision["action"] == "merge":
            session.run(
                """
                MATCH (e:Entity {name: $target})
                MERGE (q:Question {text: $question})
                MERGE (e)-[:RAISED_BY_QUESTION]->(q)
                SET e.also_known_as = coalesce(e.also_known_as, []) + $new_name
                """,
                target=decision["target"], question=question, new_name=entity_name
            )
            print(f"  ➡️  MERGED into '{decision['target']}' — {decision['reason']}")

        else:
            session.run(
                """
                MERGE (u:User {id: $user_id})
                MERGE (e:Entity {name: $entity_name})
                SET e.type = $entity_type, e.embedding = $embedding
                MERGE (u)-[:ASKED_ABOUT]->(e)
                MERGE (q:Question {text: $question})
                MERGE (e)-[:RAISED_BY_QUESTION]->(q)
                """,
                user_id=user_id, entity_name=entity_name, entity_type=entity_type,
                embedding=embedding, question=question
            )
            if decision["action"] == "link":
                session.run(
                    """
                    MATCH (a:Entity {name: $a}), (b:Entity {name: $b})
                    MERGE (a)-[r:POSSIBLY_SAME_AS]->(b)
                    SET r.reason = $reason, r.needs_review = true
                    """,
                    a=entity_name, b=decision["target"], reason=decision["reason"]
                )
                print(f"  🔗 CREATED + LINKED to '{decision['target']}' — {decision['reason']}")
            else:
                print(f"  ✨ CREATED as new — {decision['reason']}")

In [14]:
# Testing both failure directions against Real Month 3 data.

test_user = "month4-day2-resolution"

with neo4j_driver.session() as session:
    session.run("MATCH (u:User {id: $uid})-[*0..2]-(n) DETACH DELETE n, u", uid=test_user)

# --- Test A: the over-merge case (Month 3's real bug) — must stay DISTINCT ---
print("\n--- TEST A: genuinely different topics (real Month 3 bug case) ---")
write_entity_resolved(test_user, "total net sales FY2024", "financial_metric",
                       "What was Apple's total net sales for fiscal year 2024?")
write_entity_resolved(test_user, "direct_customer_segments", "strategic_question",
                       "Which customers does Apple sell directly to?")

# --- Test B: the under-merge case (Day 1's new problem) — must RESOLVE to one ---
print("\n--- TEST B: same topic, different wording ---")
write_entity_resolved(test_user, "Services segment revenue FY2024", "financial_metric",
                       "What was Apple's Services revenue in FY2024?")
write_entity_resolved(test_user, "Apple Services revenue for fiscal 2024", "financial_metric",
                       "How much did Services bring in during FY2024?")

# --- Test C: genuinely ambiguous — must LINK, not guess ---
print("\n--- TEST C: borderline pair ---")
write_entity_resolved(test_user, "capital return program", "financial_metric",
                       "What did Apple return to shareholders in FY2024?")

# --- Final state ---
with neo4j_driver.session() as session:
    entities = list(session.run(
        "MATCH (u:User {id: $uid})-[:ASKED_ABOUT]->(e:Entity) RETURN e.name AS name, e.also_known_as AS aka",
        uid=test_user))
    links = list(session.run(
        "MATCH (a:Entity)-[r:POSSIBLY_SAME_AS]->(b:Entity) RETURN a.name AS a, b.name AS b, r.reason AS reason"))

print(f"\n{'='*60}\nFINAL GRAPH STATE\n{'='*60}")
for e in entities:
    aka = f"  (also: {e['aka']})" if e["aka"] else ""
    print(f"  • {e['name']}{aka}")
print(f"\nUnresolved links flagged for review: {len(links)}")
for l in links:
    print(f"  🔗 '{l['a']}' ?= '{l['b']}' — {l['reason']}")

Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: property key does not exist. The property `embedding` does not exist in database `c44e8f50`. Verify that the spelling is correct.', position=<SummaryInputPosition line=3, column=54, offset=123>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 123, 'line': 3, 'column': 54}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\n            MATCH (u:User {id: $user_id})-[:ASKED_ABOUT]->(e:Entity)\n            RETURN e.name AS name, e.type AS type, e.embedding AS embedding\n            '



--- TEST A: genuinely different topics (real Month 3 bug case) ---
  ✨ CREATED as new — No existing entities
  🔍 Best name-similarity candidate: 'total net sales FY2024' (0.421)
  ✨ CREATED as new — Low similarity to everything (0.421)

--- TEST B: same topic, different wording ---
  🔍 Best name-similarity candidate: 'total net sales FY2024' (0.640)
  ✨ CREATED as new — Low similarity to everything (0.640)
  🔍 Best name-similarity candidate: 'Services segment revenue FY2024' (0.768)
  🔗 Shared graph neighbourhood: 0.000
  🧑‍⚖️ LLM tiebreak: SAME — Both refer to the revenue generated by Apple's Services (services segment) in fiscal year 2024.
  🔗 CREATED + LINKED to 'Services segment revenue FY2024' — Ambiguous (sim 0.768, LLM SAME) — linked, not merged

--- TEST C: borderline pair ---
  🔍 Best name-similarity candidate: 'direct_customer_segments' (0.396)


Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: property key does not exist. The property `also_known_as` does not exist in database `c44e8f50`. Verify that the spelling is correct.', position=<SummaryInputPosition line=1, column=79, offset=78>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 78, 'line': 1, 'column': 79}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'MATCH (u:User {id: $uid})-[:ASKED_ABOUT]->(e:Entity) RETURN e.name AS name, e.also_known_as AS aka'


  ✨ CREATED as new — Low similarity to everything (0.396)

FINAL GRAPH STATE
  • direct_customer_segments
  • Services segment revenue FY2024
  • total net sales FY2024
  • Apple Services revenue for fiscal 2024
  • capital return program

Unresolved links flagged for review: 1
  🔗 'Apple Services revenue for fiscal 2024' ?= 'Services segment revenue FY2024' — Ambiguous (sim 0.768, LLM SAME) — linked, not merged
